The below code is a one-time preprocessing pipeline for the AI Mental Health Companion chatbot. It takes the two source book PDFs, extracts and cleans the raw text, splits it into chunks,generates sentence embeddings for each chunk using a SentenceTransformer model, and saves everything to disk as combined_chunks.pkl and embeddings.npy — the two files the chatbot notebook needs to run. You only need to run this once. After that, just use the output files.

In [ ]:
!pip install PyMuPDF


In [ ]:
import pandas as pd
import numpy as np
import fitz

def extract_text(pdf_path, start_page, stop_page):
  text=""
  with fitz.open(pdf_path) as pdf:
    for page_num in range(start_page, stop_page):
      page= pdf[page_num]
      text+=page.get_text("text")
  return text

psy_disorder= extract_text(pdf_path="/content/Fundamentals-of-Psychological-Disorders.pdf", start_page=29, stop_page=339)
human_emo= extract_text(pdf_path="/content/Psychology_of_Human_Emotion.pdf", start_page=21, stop_page=900)

import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|https\S+', '', text)             # remove URLs
    text = re.sub(r'\s+', ' ', text)                         # normalize spaces
    text = re.sub(r'\b\d+\b', '', text)                      # remove standalone numbers
    text = re.sub(r'figure\s*\d+|table\s*\d+', '', text)     # remove "figure 1" etc.
    text = re.sub(r'[^\x00-\x7f]', ' ', text)                # remove non-ASCII
    text = re.sub(r'\([a-zA-Z\s,&.]*\d{4}\)', '', text)      # remove (Author, 2024)
    text = re.sub(r'[^a-zA-Z0-9.,!?;:\'\"()\-\s]', '', text) # keep important punctuation
    text = re.sub(r'\s+', ' ', text).strip()                 # removes extra spaces
    return text

clean_psy_disorder= clean_text(psy_disorder)
clean_human_emo= clean_text(human_emo)

clean_psy_disorder[:500]

clean_human_emo[:500]

import nltk
from nltk.tokenize import sent_tokenize
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


sentence_disorder = sent_tokenize(clean_psy_disorder)
sentence_emo = sent_tokenize(clean_human_emo)

from nltk.tokenize import word_tokenize

def chunk_sentence(sentences, max_tokens=400):
    """
    tokenizing in words and chunking:
    - If a sentence has <= max_tokens, simply merge.
    - If a sentence > max_tokens, split it into multiple pieces of max_tokens each.
    """
    chunks = []

    for sent in sentences:
        tokens = word_tokenize(sent)

        if len(tokens) <= max_tokens:
            chunks.append(sent)
        else:
            # Split long sentence into max_tokens pieces
            for i in range(0, len(tokens), max_tokens):
                chunks.append(" ".join(tokens[i:i+max_tokens]))

    return chunks

chunk_disorder = chunk_sentence(sentence_disorder, max_tokens=400)
chunk_emo = chunk_sentence(sentence_emo, max_tokens=400)
combined_chunk= chunk_disorder + chunk_emo
print(f"Total chunks disorder book: {len(chunk_disorder)}")
print(f"Total chunks emotions book: {len(chunk_emo)}")

print("\nSample chunk (Emotion book):", chunk_emo[0])

import pickle
from google.colab import files

# FIX: filename was "chunks.pkl" but download was trying "combined_chunk.pkl" — both now consistent
with open("combined_chunks.pkl", "wb") as f:
    pickle.dump(combined_chunk, f)

print("Combined chunks saved successfully as 'combined_chunks.pkl'!")

# Download the chunks file
files.download("combined_chunks.pkl")

embedder = SentenceTransformer('all-MiniLM-L6-v2')

embeddings= embedder.encode(combined_chunk, show_progress_bar=True)

print("Embedding shape:", embeddings.shape)
print("Example embedding vector (first chunk):")
print(embeddings[0][:10])
np.save("embeddings.npy", embeddings)
files.download("embeddings.npy")
